In [1]:
include("./trajopt/utils.jl")
include("./trajopt/dynamics.jl")
include("./funlopt/funl_dynamics.jl")
include("./funlopt/funl_utils.jl")
include("./funlopt/funl_constraint.jl")
include("./trajopt/scaling.jl")

compute_scaling (generic function with 1 method)

In [2]:
c = [1; 0]

Q1 = [1 0;0 1]
a1 = -1

d2 = [1; 0]

A = [0 1]
b = [0.5]

Atilde = [1 zeros(1,2); 0 A]
bbar = [1; b]
E = [0 1 0;0 0 1]
e1 = [1;0;0]

3-element Vector{Int64}:
 1
 0
 0

In [22]:
model = Model(Mosek.Optimizer)

A JuMP Model
Feasibility problem with:
Variables: 0
Model mode: AUTOMATIC
CachingOptimizer state: EMPTY_OPTIMIZER
Solver name: Mosek

In [29]:
Xbar = @variable(model, [1:3, 1:3], PSD)
# @constraint(model, tr(Xbar) >= 0)

3×3 Symmetric{VariableRef, Matrix{VariableRef}}:
 _[7]   _[8]   _[10]
 _[8]   _[9]   _[11]
 _[10]  _[11]  _[12]

In [30]:
@constraint(model, Atilde * Xbar * Atilde' .== bbar * bbar')

2×2 Matrix{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.EqualTo{Float64}}, ScalarShape}}:
 _[7] = 1     _[10] = 0.5
 _[10] = 0.5  _[12] = 0.25

In [31]:
@constraint(model,tr(Q1*E*Xbar*E') + a1 >= 0)
@constraint(model, d2' * E * Xbar * e1 >= 0)
@constraint(model, e1' * Xbar * e1 == 1)

_[7] = 1

In [45]:
weight = 1e8
@objective(model,Min,c'*E*Xbar*e1 + weight * tr(Xbar[2:3,2:3]))

_[8] + 100000000 _[9] + 100000000 _[12]

In [46]:
optimize!(model)
@show termination_status(model)

Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 15              
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 6               
  Matrix variables       : 1 (scalarized: 6)
  Integer variables      : 0               

Optimizer started.
Optimizer terminated. Time: 0.00    

termination_status(model) = MathOptInterface.OPTIMAL


OPTIMAL::TerminationStatusCode = 1

In [47]:
value.(Xbar)

3×3 Matrix{Float64}:
 1.0        0.0184266   0.5
 0.0184266  0.75        0.00921513
 0.5        0.00921513  0.25

In [38]:
tr(Q1*E*value.(Xbar)*E') + a1

2.3046489161515638e-9

In [ ]:
# @constraint(model, Atilde * Xbar * Atilde' .== bbar * bbar')
Atilde * value.(Xbar) * Atilde' - bbar * bbar'

In [ ]:
Atilde

In [ ]:
bbar * bbar'

In [ ]:
value.(Xbar)